# 01 — Théorie et prototype

**Projet 1 — Groupe 1 : solveur de Nash m×n et duopoles (Cournot, Bertrand).**

S2 : faire tourner les fonctions et retrouver les résultats du cours.
Lancer depuis la racine du dépôt (`jupyter notebook`).

In [1]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))  # racine du dépôt

import numpy as np
from src.jeux import (
    DILEMME_A, DILEMME_B, BATAILLE_A, BATAILLE_B, PFC_A, PFC_B,
    enumeration_supports, est_nash, equilibres_nash_purs, cournot, cournot_grille, bertrand,
)
print('NumPy', np.__version__)

NumPy 2.5.4


## 1. Jeux de référence du cours

On doit retrouver :
- dilemme du prisonnier : (Trahir, Trahir) ;
- bataille des sexes : 3 équilibres dont le mixte (2/3, 1/3) ;
- pierre-feuille-ciseaux : (1/3, 1/3, 1/3).

In [2]:
jeux = {
    'Dilemme du prisonnier': (DILEMME_A, DILEMME_B),
    'Bataille des sexes': (BATAILLE_A, BATAILLE_B),
    'Pierre-feuille-ciseaux': (PFC_A, PFC_B),
}
for nom, (A, B) in jeux.items():
    eqs = enumeration_supports(A, B)
    print(f'\n{nom} : {len(eqs)} équilibre(s)')
    for x, y in eqs:
        print('  x =', np.round(x, 4), ' y =', np.round(y, 4),
              ' non-déviation OK =', est_nash(A, B, x, y))


Dilemme du prisonnier : 1 équilibre(s)
  x = [0. 1.]  y = [0. 1.]  non-déviation OK = True

Bataille des sexes : 3 équilibre(s)
  x = [1. 0.]  y = [1. 0.]  non-déviation OK = True
  x = [0. 1.]  y = [0. 1.]  non-déviation OK = True
  x = [0.6667 0.3333]  y = [0.3333 0.6667]  non-déviation OK = True

Pierre-feuille-ciseaux : 1 équilibre(s)
  x = [0.3333 0.3333 0.3333]  y = [0.3333 0.3333 0.3333]  non-déviation OK = True


## 2. Équilibres purs

Le dilemme du prisonnier n'a qu'un équilibre pur : (Trahir, Trahir) = indices (1, 1).

In [3]:
print('Équilibres purs (dilemme) :', equilibres_nash_purs(DILEMME_A, DILEMME_B))

Équilibres purs (dilemme) : [(1, 1)]


## 3. Cournot : formule fermée vs grille

Avec a=100, b=1, c=10 : q* = 30 par firme, profit 900 ; cartel 22,5 et 1012,5.

In [4]:
r = cournot(a=100, b=1, c=10, n=2)
for k, v in r.items():
    print(k, ':', v)

print('\nÉquilibres de Cournot sur grille entière (qmax=100) :',
      cournot_grille(100, 1, 10, qmax=100))
print('Bertrand :', bertrand(100, 1, 10))

n : 2
quantite_par_firme : 30.0
quantite_totale : 60.0
prix : 40.0
profit_par_firme : 900.0
cartel : {'quantite_par_firme': 22.5, 'quantite_totale': 45.0, 'prix': 55.0, 'profit_par_firme': 1012.5, 'profit_total': 2025.0}

Équilibres de Cournot sur grille entière (qmax=100) : [(29.0, 31.0), (30.0, 30.0), (31.0, 29.0)]
Bertrand : {'prix_equilibre': 10.0, 'profit_par_firme': 0.0, 'prix_monopole': 55.0, 'profit_monopole': 2025.0}


## 4. Itération des meilleures réponses (contraction 1/2)

Vérification : la suite q_{t+1} = (a − c − b q_t) / (2b) converge vers q* = 30.

In [5]:
a, b, c = 100.0, 1.0, 10.0
q = np.array([0.0, (a - c) / b])
for t in range(10):
    q = (a - c - b * q[::-1]) / (2 * b)
    print(f'itération {t+1}: q = {np.round(q, 4)}')
print('q* =', np.round(q, 6))

itération 1: q = [ 0. 45.]
itération 2: q = [22.5 45. ]
itération 3: q = [22.5  33.75]
itération 4: q = [28.125 33.75 ]
itération 5: q = [28.125  30.9375]
itération 6: q = [29.5312 30.9375]
itération 7: q = [29.5312 30.2344]
itération 8: q = [29.8828 30.2344]
itération 9: q = [29.8828 30.0586]
itération 10: q = [29.9707 30.0586]
q* = [29.970703 30.058594]
